#**GT Deep Learning - Case Study**
This script is proposed to be a case study aiming to classify the presence of hypersaline tidal flats in satellite imagery.
This scripts was building by Solved interprise but change and adapteted by Geodatin enterprise


# Setup software libraries (Instalação e importação de biliotecas)

Install needed libraries to the notebook VM.  Authenticate as necessary.

In [ ]:
# Cloud authentication.
from google.colab import auth
auth.authenticate_user()
instalar = True

In [ ]:
# Necessary API's install to notebook VM.
if instalar:
  !pip install earthengine-api
  !pip install oauth2client
  !pip install google-api-python-client
  !pip install pygeoj
  instalar = False

import json
import logging
import os
logging.getLogger('googleapicliet.discovery_cache').setLevel(logging.ERROR)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
pathDrive = '/content/drive/MyDrive/energias'
!ls '/content/drive/MyDrive/energias'

 amostra_Normalizado_GT_DL_AfloramentoCaatV5.ipynb
 ClassificationDL_29_11_2022.pptx
 conect_drive_tobucket.ipynb
 dict_stat_nyearRegComp.json
'Emenda do curso de Python em SELPER.gdoc'
'guia de conteudo criado com gemini.gdoc'
'Introdução a DL Semana Medio Amb.pptx'
'Introdução a DL Semana Meio Ambiente PPGM.pptx'
 makeAmostra_Normalizado_GT_DL_fotoV.ipynb
 makeMaping_with_predictPatchs.ipynb
 model
 PedrictUnetAttenty_fotovoltaicaCaat.ipynb
 registros
 treinamento_GT_DL_fotovoltaicaCaat.ipynb
 treinamentoSentinel2_GT_DL_AfloramentoCaat.ipynb


In [ ]:
# Import, authenticate and initialize the Earth Engine library.
import ee
ee.Authenticate()
ee.Initialize(project='geo-data-s') #

In [ ]:
# Tensorflow setup.
import os
# %tensorflow_version 1.x
import tensorflow as tf
print("We working with TF Version = ", tf.__version__)
# tf.enable_eager_execution()

device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  raise SystemError('GPU device not found')
print('Found GPU at: {}'.format(device_name))

We working with TF Version =  2.17.1
Found GPU at: /device:GPU:0


In [ ]:
# Folium setup.
import folium
print(folium.__version__)

# Define the URL format used for Earth Engine generated map tiles.
EE_TILES = 'https://earthengine.googleapis.com/map/{mapid}/{{z}}/{{x}}/{{y}}?token={token}'

0.19.2


# Set other global variables (Parâmetros Globais)

In [ ]:
# General
GOAL_CLASS     = 'fotovoltaica'
VERSION        = '2';
GDRIVE         = 'unet-'+ GOAL_CLASS
FOLDER_TRAIN   = 'train_chips_fotoV1'
FOLDER_TEST    = 'eval_chips_fotoV1'


#Local paths
BASE_PATH      = pathDrive + '/ROIs'
LOCAL_PATH     =  BASE_PATH  + '/unet-' + GOAL_CLASS
MODEL_DIR      =  LOCAL_PATH + '/checkpoint/v' + VERSION
OUTPUT_PATH    =  LOCAL_PATH + '/output/v' + VERSION
biomes = 'CAATINGA'
# Exportation Configs

FOLDER_patch   = 'allPatch'
FOLDER_classification = 'mb7_'+ GOAL_CLASS + '_' + VERSION
asset_limitCaat = 'users/CartasSol/shapes/nCaatingaBff3500'
# asset_SHPfoto = 'projects/mapbiomas-workspace/AMOSTRAS/col9/CAATINGA/Energias/shp_manual_fotovoltaicas_Br_2024'
# asset_SHPfoto = 'projects/mapbiomas-workspace/AMOSTRAS/col9/CAATINGA/Energias/polygons/polygons_base_paneis_fotovoltaicos_19_11'
asset_SHPfoto = 'projects/mapbiomas-workspace/AMOSTRAS/col9/CAATINGA/Energias/polygons/polygons_base_paneis_fotovoltaicos_08_12'
asset_fotovol_pred = 'projects/geo-data-s/assets/fotovoltaica/versao1'
regions_on_paneis_foto =  "projects/mapbiomas-workspace/AMOSTRAS/col9/CAATINGA/Energias/polygons/regions_on_paneis_fotovoltaicos_07_12joins_corr"
asset_ptos_bom_result = "projects/mapbiomas-workspace/AMOSTRAS/col9/CAATINGA/Energias/polygons/pontos_useLabel_fotovoltaicos_08_12"
asset_areas_erros = "projects/mapbiomas-workspace/AMOSTRAS/col9/CAATINGA/Energias/polygons/regions_erro_predict_fotovoltaicos_08_12"

asset_raster_biomas = 'projects/mapbiomas-workspace/AUXILIAR/biomas-raster-41'
bandMosaic = ['B','G','R','N']
bandMos = ['B','G','R','N']


indexBands   = ['B','G','R','N','iia','evi','ri','gli','shape']

BANDS          = indexBands
RESPONSE       = 'supervised'
FEATURES       = BANDS + [RESPONSE]
assetNICFI =  'projects/planet-nicfi/assets/basemaps/americas'
# Specify the size and shape of patches expected by the model.
KERNEL_SIZE  = 256
KERNEL_SHAPE = [KERNEL_SIZE, KERNEL_SIZE]
COLUMNS = [
  tf.io.FixedLenFeature(shape=KERNEL_SHAPE, dtype=tf.float32) for k in FEATURES
]
FEATURES_DICT = dict(zip(FEATURES, COLUMNS))

# Sizes of the training and evaluation datasets.
TRAIN_SIZE = 0
EVAL_SIZE  = 0

# Specify model training parameters.
BATCH_SIZE  = 12
DROPOUT     = 0.3
EPOCHS      = 50
BUFFER_SIZE = 1000
OPTIMIZER   = 'Nadam'
LOSS        = 'BinaryCrossentropy'
METRICS     = ['RootMeanSquaredError','Hinge']

# Imagery

Gather and setup the imagery to use for inputs (predictors).  This is a three-year, cloud-free, Landsat 8 composite.  Display it in the notebook for a sanity check.

In [ ]:
# Bare Soil Index
def CalculateIndiceAllinOne(img):

    ndwi = (img.expression(
            "float(b('nir') - b('swir2')) / (b('nir') + b('swir2'))")
                .add(1).multiply(10000).rename(['ndwi']).toInt16()
        )
    awei = (img.expression(
                "float(4 * (b('green') - b('swir2')) - (0.25 * b('nir') + 2.75 * b('swir1')))"
            ).add(1).multiply(10000).rename("awei")).toInt16()
    # LSWI	Land Surface Water Index
    lswi = img.expression(
            "float((b('nir') - b('swir1')) / (b('nir') + b('swir1')))")\
                .rename(['lswi']).toFloat()

    evi2 = (img.expression(
        "float(2.4 * (b('N') - b('R')) / (1 + b('N') + b('R')))")
            .add(1).multiply(10000).rename(['evi']).toInt16()
    )
    # UI	Urban Index	urban
    ui = nImg.expression(
            "float((b('swir2') - b('nir')) / (b('swir2') + b('nir')))")\
                .rename(['ui'])
    ui = ui.add(1).multiply(10000).toUint16()
    # MBI	Modified Bare Soil Index
    mbi = nImg.expression(
            "float(((b('swir1') - b('swir2') - b('nir')) /" +
                " (b('swir1') + b('swir2') + b('nir'))) + 0.5)")
                    .rename(['mbi']
        )
    mbi = mbi.add(1).multiply(10000).toUint16()

    gli = (img.expression(
        "float((2 * b('G') - b('R') - b('B')) / (2 * b('G') + b('R') + b('B')))")
            .add(1).multiply(10000).rename(['gli']).toInt16()
    )


    return (img.addBands(ndwi).addBands(awei)
                .addBands(lswi).addBands(evi2)
                .addBands(ui).addBands(mbi))



In [ ]:
def process_normalized_img (imgA):
    return imgA.toFloat().divide(10000).set('year', imgA.get('year'))

In [ ]:
base_biomas = ee.Image(asset_raster_biomas).gt(0)
polygonsSHPfotoV = ee.FeatureCollection(asset_SHPfoto)
# construindo a camada de reference ou label
label_map = polygonsSHPfotoV.reduceToImage(['class'], ee.Reducer.first()).unmask(0).toByte().rename(RESPONSE);
print("  ", label_map.bandNames().getInfo())
print(f"we have {polygonsSHPfotoV.size().getInfo()} polygon ")

   ['supervised']
we have 484 polygon 


In [ ]:
ptos_selectReg = ee.FeatureCollection(asset_ptos_bom_result)
print(f" we have {ptos_selectReg.size().getInfo()} ptos with good label in region ")

regions_FV = ee.FeatureCollection(regions_on_paneis_foto)
print(f' we have {regions_FV.size().getInfo()} regions')
print(regions_FV.first().getInfo())

# 'idCod'
regions_FVselect = regions_FV.filterBounds(ptos_selectReg)
print(regions_FVselect.size().getInfo())

lstIDsSel = regions_FVselect.reduceColumns(ee.Reducer.toList(), ['idCod']).get('list').getInfo()
print(len(lstIDsSel))
print(lstIDsSel[:2])
lstIDsSelect = [kk + '_pred' for kk in lstIDsSel]

 we have 44 ptos with good label in region 
 we have 120 regions
{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-53.31394857223446, -27.693474504531213], [-53.300740768299, -27.693474533794873], [-53.300740768299, -27.681626652871785], [-53.31394857223446, -27.681626648650777], [-53.31394857223446, -27.693474504531213]]]}, 'id': '00000000000000000000', 'properties': {'idCod': '00000000000000000000', 'region': 'fotovoltaica'}}
39
39
['00000000000000000008', '00000000000000000013']


In [ ]:
# adding the other label class
label1verFV = ee.ImageCollection(asset_fotovol_pred)
print(f" label pred  with {label1verFV.size().getInfo()} patches")
print("show the first Id ", label1verFV.first().id().getInfo())
# filtering the regions selected
label1raVerFV = label1verFV.filterBounds(ptos_selectReg)

print(f" size of labels regions selected {label1raVerFV.size().getInfo()}")
label1raVerFV = label1raVerFV.map(lambda img: img.toByte());  # selfMask()
label1raVerFV = ee.Image(label1raVerFV.max()).toInt16()

areas_erros = ee.FeatureCollection(asset_areas_erros).map(lambda feat: feat.set('etiqueta', 1))
label_mapErr = areas_erros.reduceToImage(['etiqueta'], ee.Reducer.first()).unmask(0).toInt16().rename('etiqueta');

label1raVerFV = label1raVerFV.unmask(0).subtract(label_mapErr.unmask(0)).eq(1)


 label pred  with 130 patches
show the first Id  00000000000000000000_pred
 size of labels regions selected 43


In [ ]:
# update Label Map
label_map = label_map.add(label1raVerFV.unmask(0)).gt(0)

In [ ]:
# testing polygons cruzamentos com os pontos de selecting
print(polygonsSHPfotoV.filterBounds(ptos_selectReg).size().getInfo() )

2


In [ ]:
assetNICFI = 'projects/planet-nicfi/assets/basemaps/americas'
imgMosPlanet = ee.ImageCollection(assetNICFI)
print("ver quantidade de image planet ", imgMosPlanet.size().getInfo())

ver quantidade de image planet  61


In [ ]:
baseClassV = '1'
lstYear = [2021,2022,2023]
yearClass  = 2024

In [ ]:
nmonths = 10

In [ ]:
nmonths = 10
imgPlanet = imgMosPlanet.filter(ee.Filter.date('2024-10-01','2024-11-20')).first()
imgPlanet =  ee.Image(imgPlanet).updateMask(base_biomas);
print("number the imagens loading ", imgPlanet.bandNames().getInfo())
# imgMosaicNorm = imgMosaic.map(lambda imgg : process_normalized_img(imgg))

# imgMosaicNorm = imgMosaicNorm.map(lambda imgg : CalculateIndiceAllinOne(imgg))
# # print("number the imagens loading ", imgMosaicNorm.getInfo())
# imgMosaicNorm = process_normalized_img(imgPlanet)
imgMosaicNorm = CalculateIndiceAllinOne(imgPlanet.toInt16())
bandsMosaicsYears = imgMosaicNorm.bandNames().getInfo()
print("With bands spectrals in list = ")
print(bandsMosaicsYears)

number the imagens loading  ['B', 'G', 'R', 'N']
With bands spectrals in list = 
['B', 'G', 'R', 'N', 'iia', 'evi', 'ri', 'gli', 'shape']


In [ ]:
minMaxDict = {
    'B_max' : 9527,
    'B_min' :  160,
    'G_max' :  9973,
    'G_min' :  328,
    'N_max' :  9988,
    'N_min' :  894,
    'R_max' :  9998,
    'R_min' :  237,
    'evi_max' :  31444,
    'evi_min' :  6272,
    'gli_max' :  15950,
    'gli_min' :  7809,
    'iia_max' :  6596,
    'iia_min' :  668,
    'ri_max' :  19050,
    'ri_min' : -2630,
    'shape_max' :  10189,
    'shape_min' :  9981,
}
dictSame = True

In [ ]:
# Reduce the region. The region parameter is the Feature geometry.
coord = [-42.8045, -15.2194]
# PointCentro = ee.Geometry.Point(coord)
imagTest = imgMosaicNorm
# print("imagTest ", imgMosaicNorm.getInfo())
if not dictSame:
    minMaxDict = imgMosaicNorm.reduceRegion(
        reducer= ee.Reducer.minMax(),
        geometry= polygonsSHPfotoV.geometry(),
        scale= 30,
        maxPixels= 1e13
    );
    minMaxDict = minMaxDict.getInfo()


#  The result is a Dictionary.  Print it.
print("dict min max ", );
for kk, vv in minMaxDict.items():
    print(kk, " = ", vv)



dict min max 
B_max  =  9527
B_min  =  160
G_max  =  9973
G_min  =  328
N_max  =  9988
N_min  =  894
R_max  =  9998
R_min  =  237
evi_max  =  31444
evi_min  =  6272
gli_max  =  15950
gli_min  =  7809
iia_max  =  6596
iia_min  =  668
ri_max  =  19050
ri_min  =  -2630
shape_max  =  10189
shape_min  =  9981


Prepare the response (what we want to predict).  This is impervious surface area (in fraction of a pixel) from the 2016 NLCD dataset.  Display to check.

In [ ]:
mapidMosaic = imagTest.getMapId(
                    {'bands': ['R', 'G', 'B'], 'min': 100, 'max': 5000})
map = folium.Map(location=[-4.9827, -36.9058 ],zoom_start=10)
folium.TileLayer(
    tiles=mapidMosaic['tile_fetcher'].url_format,
    attr='Planet',
    overlay=True,
    name='Mosaic composite',
  ).add_to(map)

mapid = label_map.selfMask().getMapId({'min': 0, 'max': 1,'palette': ["ff0000"]})
# mapid = label1raVerFV.selfMask().getMapId({'min': 0, 'max': 1,'palette': ["ff0000"]})
folium.TileLayer(
    tiles=mapid['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name='nlcd impervious',

  ).add_to(map)
map.add_child(folium.LayerControl())
map

Stack the 2D images (Landsat composite and NLCD impervious surface) to create a single image from which samples can be taken.  Convert the image into an array image in which each pixel stores 256x256 patches of pixels for each band.  This is a key step that bears emphasis: to export training patches, convert a multi-band image to [an array image](https://developers.google.com/earth-engine/arrays_array_images#array-images) using [`neighborhoodToArray()`](https://developers.google.com/earth-engine/api_docs#eeimageneighborhoodtoarray), then sample the image at points.

In [ ]:
def return_array_frommosaic(imgMosInd, layer_label):

    featureStack = ee.Image.cat([
                    imgMosInd.unmask(0),  layer_label.unmask(0)]).float()

    list = ee.List.repeat(1, KERNEL_SIZE)
    lists = ee.List.repeat(list, KERNEL_SIZE)
    kernel = ee.Kernel.fixed(KERNEL_SIZE, KERNEL_SIZE, lists)
    arrays = featureStack.neighborhoodToArray(kernel)

    return arrays

Use some pre-made geometries to sample the stack in strategic locations.  Specifically, these are hand-made polygons in which to take the 256x256 samples.  Display the sampling polygons on a map, red for training polygons, blue for evaluation.

In [ ]:
shpGeral = ee.FeatureCollection(polygonsSHPfotoV)
# merge with more 35 feature Reg
shpGeral = shpGeral.merge(regions_FVselect)
# randomize
shpGeral = shpGeral.randomColumn('random')
print("Foram carregados {} polygons para treinamento e teste".format(shpGeral.size().getInfo()))


Foram carregados 523 polygons para treinamento e teste


# Sampling (Subdivisão Amostral em Treino e Teste)

The mapped data look reasonable so take a sample from each polygon and merge the results into a single export.  The key step is sampling the array image at points, to get all the pixels in a 256x256 neighborhood at each point.  It's worth noting that to build the training and testing data for the FCNN, you export a single TFRecord file that contains patches of pixel values in each record.  You do NOT need to export each training/testing patch to a different image.  Since each record potentially contains a lot of data (especially with big patches or many input bands), some manual sharding of the computation is necessary to avoid the `computed value too large` error.  Specifically, the following code takes multiple (smaller) samples within each geometry, merging the results to get a single export.

In [ ]:
trainingPolys = shpGeral.filter(ee.Filter.lte('random', 0.8))
evalPolys     = shpGeral.filter(ee.Filter.gt('random', 0.8))

print("Number of training polygons: "+ str(trainingPolys.size().getInfo()))
print("Number of evalluation/test polygons: "+ str(evalPolys.size().getInfo()))

# polyImage = ee.Image(0).byte().paint(trainingPolys, 1).paint(evalPolys, 2)
# polyImage = polyImage.updateMask(polyImage)

# mapid = polyImage.getMapId({'min': 1, 'max': 2, 'palette': ['red', 'blue']})
# map = folium.Map(location=[-6.1148, -40.9686], zoom_start=9)
# folium.TileLayer(
#     tiles=mapid['tile_fetcher'].url_format,
#     attr='Google Earth Engine',
#     overlay=True,
#     name='training polygons',
#   ).add_to(map)
# map.add_child(folium.LayerControl())
# map

Number of training polygons: 427
Number of evalluation/test polygons: 96


## Train/Test Exportation (Exportação de Conjunto de Dados de Treino e Teste)

In [ ]:
# version_samples_acc = "_v1"
imgMosaicNorm = imgMosaicNorm.select(indexBands)

# Convert the feature collections to lists for iteration.
trainingPolysList = trainingPolys.toList(trainingPolys.size())
evalPolysList     = evalPolys.toList(evalPolys.size())

numero_pol_train = trainingPolysList.size().getInfo()
numero_pol_test = evalPolysList.size().getInfo()
print("lista de Poygons training ", numero_pol_train)
print("lista de Poygons evaluate ", numero_pol_test)

# These numbers determined experimentally.
n = 10 # Number of shards in each polygon.
N = 5 # Total sample size in each polygon.

#Add some generalism
TRAIN_SIZE = numero_pol_train * N
EVAL_SIZE = numero_pol_test * N
print('TRAIN:' + str(TRAIN_SIZE))
print('EVAL:' + str(EVAL_SIZE))

lista de Poygons training  427
lista de Poygons evaluate  96
TRAIN:2135
EVAL:480


In [ ]:
# juntando os dois label maps
label_map = label_map.unmask(0)

In [ ]:
def export_imageto_TFrecord(geomSSample, ngtr, CCountTrain, nFOLDER, sufix):
    desc =   sufix + "_fotoV_YY_" + str(yearClass) + "_" + str(nmonths) + '_g' + str(ngtr) + "_" + str(CCountTrain)
    task = ee.batch.Export.table.toDrive(
              collection = geomSSample,
              description = desc,
              folder =  nFOLDER, # + version_samples_acc,   # pathDrive + '/' +
              fileNamePrefix = desc,
              fileFormat = 'TFRecord',
              selectors = indexBands + [RESPONSE]
            )
    task.start()
    print("export TFRecord =>> " + desc + "   in " + nFOLDER)

In [ ]:
# este aumentaram o número de poligos e estão aproveitando os
# labels da versão 1 em áreas seleccionadas
FOLDER_TRAIN   = 'train_chips_fotoV3'
FOLDER_TEST    =  'eval_chips_fotoV3'

In [ ]:
# imgMosaicYear = imgMosaicYear.divide(10000).toFloat()
array_mosaic = return_array_frommosaic(imgMosaicNorm, label_map)
print("array mosaic ", array_mosaic.bandNames().getInfo())

for gtr in range(numero_pol_train):
    print("processing poligons training", gtr)
    geomSample = ee.FeatureCollection([])
    countTrain = 0
    for i in range(n):
        sample = array_mosaic.sample(
                    region = ee.Feature(trainingPolysList.get(gtr)).geometry(),
                    scale = 4,
                    numPixels = N, # Size of the shard.  / n
                    seed = i,
                    # tileScale = 8
                )
        geomSample = geomSample.merge(sample)
        countTrain += 1
    print("Exporting {} point features ".format(geomSample.size().getInfo()))
    export_imageto_TFrecord(ee.FeatureCollection(geomSample), gtr, countTrain, FOLDER_TRAIN, 'train' )



array mosaic  ['B', 'G', 'R', 'N', 'iia', 'evi', 'ri', 'gli', 'shape', 'supervised']
processing poligons training 0
Exporting 50 point features 
export TFRecord =>> train_fotoV_YY_2024_10_g0_10   in train_chips_fotoV3
processing poligons training 1
Exporting 50 point features 
export TFRecord =>> train_fotoV_YY_2024_10_g1_10   in train_chips_fotoV3
processing poligons training 2
Exporting 50 point features 
export TFRecord =>> train_fotoV_YY_2024_10_g2_10   in train_chips_fotoV3
processing poligons training 3
Exporting 50 point features 
export TFRecord =>> train_fotoV_YY_2024_10_g3_10   in train_chips_fotoV3
processing poligons training 4
Exporting 50 point features 
export TFRecord =>> train_fotoV_YY_2024_10_g4_10   in train_chips_fotoV3
processing poligons training 5
Exporting 50 point features 
export TFRecord =>> train_fotoV_YY_2024_10_g5_10   in train_chips_fotoV3
processing poligons training 6
Exporting 50 point features 
export TFRecord =>> train_fotoV_YY_2024_10_g6_10   in tra

In [ ]:
# Export all the evaluation data.
for gtr in range(evalPolys.size().getInfo()):
    print("processing poligons evaluate", gtr)
    geomSample = ee.FeatureCollection([])
    countTest = 0
    for i in range(n):
        sample = array_mosaic.sample(
          region = ee.Feature(evalPolysList.get(gtr)).geometry(),
          scale = 4,
          numPixels = N,
          seed = i,
        #   tileScale = 8
        )
        geomSample = geomSample.merge(sample)
        countTest += 1
    print("Exporting {} point features ".format(geomSample.size().getInfo()))
    export_imageto_TFrecord(ee.FeatureCollection(geomSample), gtr, countTest, FOLDER_TEST, 'test')

processing poligons evaluate 0
Exporting 50 point features 
export TFRecord =>> test_fotoV_YY_2024_10_g0_10   in eval_chips_fotoV3
processing poligons evaluate 1
Exporting 50 point features 
export TFRecord =>> test_fotoV_YY_2024_10_g1_10   in eval_chips_fotoV3
processing poligons evaluate 2
Exporting 50 point features 
export TFRecord =>> test_fotoV_YY_2024_10_g2_10   in eval_chips_fotoV3
processing poligons evaluate 3
Exporting 50 point features 
export TFRecord =>> test_fotoV_YY_2024_10_g3_10   in eval_chips_fotoV3
processing poligons evaluate 4
Exporting 50 point features 
export TFRecord =>> test_fotoV_YY_2024_10_g4_10   in eval_chips_fotoV3
processing poligons evaluate 5
Exporting 50 point features 
export TFRecord =>> test_fotoV_YY_2024_10_g5_10   in eval_chips_fotoV3
processing poligons evaluate 6
Exporting 50 point features 
export TFRecord =>> test_fotoV_YY_2024_10_g6_10   in eval_chips_fotoV3
processing poligons evaluate 7
Exporting 50 point features 
export TFRecord =>> tes



---



In [ ]:
from google.colab import drive
drive.mount('/content/drive')